# Machine Learning Lab: Classification Pipeline Deep Dive
This lab executes a comprehensive Machine Learning pipeline using Pandas, Seaborn, and Scikit-Learn to classify breast cancer tumors as Malignant or Benign. The workflow includes EDA, preprocessing, model training, cross-validation, and hyperparameter tuning.

In [ ]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("All libraries imported successfully!")

### Step 2: Exploratory Data Analysis (EDA)
Load the data and analyze feature distributions and target balance.

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target # 0 = Malignant, 1 = Benign

display(df.head())
print("\nMissing Values:\n", df.isnull().sum().sum())

In [ ]:
# Visualize the balance of our target classes
plt.figure(figsize=(6,4))
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Distribution of Target Classes (0=Malignant, 1=Benign)')
plt.show()

### Step 3: Data Preprocessing
Data is split into training and testing cohorts. Features are scaled using StandardScaler to ensure distance-based algorithms (like KNN) operate correctly.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")

### Step 4: Model Training and Cross-Validation
Train a Logistic Regression model and use K-Fold cross-validation to ensure the accuracy score is robust across different data splits.

In [ ]:
# Logistic Regression Training
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)
log_preds = log_model.predict(X_test_scaled)

# Evaluation
print("--- Logistic Regression ---")
print(f"Test Accuracy: {accuracy_score(y_test, log_preds):.4f}")
print("\nClassification Report:\n", classification_report(y_test, log_preds))

# 5-Fold Cross Validation
cv_scores = cross_val_score(log_model, X_train_scaled, y_train, cv=5)
print(f"Cross-Validation Accuracies (5 folds): {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")

### Step 5: Advanced Evaluation (Confusion Matrix)
Visualizing the confusion matrix to understand False Positives vs False Negatives.

In [ ]:
cm = confusion_matrix(y_test, log_preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

### Step 6: Hyperparameter Tuning (Grid Search)
Use GridSearchCV to automatically test multiple hyperparameters and find the optimal configuration for a Decision Tree.

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42)

# Define the parameter grid to search over
param_grid = {
    'max_depth': [2, 3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

# Execute Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(tree_model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

print("--- Decision Tree Tuning ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

# Evaluate the best model on the unseen test set
best_tree = grid_search.best_estimator_
tree_preds = best_tree.predict(X_test_scaled)
print(f"\nTest Set Accuracy (using best params): {accuracy_score(y_test, tree_preds):.4f}")